In [26]:
# Exercise 1 — Train/Test + Linear Regression


#Task:
#Predict y from X

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

# Data
import numpy as np
import pandas as pd

df = pd.DataFrame({
    "feature1": [1,2,3,4,5,6],
    "feature2": [10,20,30,40,50,60],
    "target": [15,25,35,45,55,65]
})

# TODO:
# 1. Split data
# 2. Train model
# 3. Predict
# 4. Compute RMSE

# Split x and y
x = df[["feature1","feature2"]]
y = df["target"]

# Train and Test Split
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.3, random_state=42)
x_train, x_test, y_train, y_test

# Train model
lin_reg = LinearRegression()
lin_reg.fit(x_train, y_train)

# Predict
y_predict = lin_reg.predict(x_test)

print(y_predict[:5])

# 5. Compute RMSE
mse = mean_squared_error(y_test, y_predict)
rmse = np.sqrt(mse)

print(rmse)

# 6. Compute SD
target_sd = df["target"].std()
print(target_sd)

#7. Compute R2
r2 = r2_score(y_test,y_predict)
print(r2)

[15. 25.]
1.3586517280808823e-14
18.708286933869708
1.0


In [31]:
# Exercise 2 — Classification + Logistic Regression

# Task:
# Predict binary outcome

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import classification_report

df = pd.read_csv('datasets/diabetes.csv')

# TODO:
# 1. Train model
# 2. Predict
# 3. Compute accuracy, precision, recall

# split data into x and y and then into train and test
x = df[['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age']]
y = df['Outcome']
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.3, random_state=42, stratify=y)

# Train Model
log =  LogisticRegression(max_iter=1000)
log.fit(x_train, y_train)

# predict
y_predict = log.predict(x_test)

# Compute accuracy, precision, recall
acc_scr = accuracy_score(y_test,y_predict)
print("Accuracy Score",acc_scr)

prec_scr = precision_score(y_test,y_predict)
print("Precision Score",prec_scr)

rec_scr = recall_score(y_test, y_predict)
print("Recall Score", rec_scr)

classification_report
cls_rep = classification_report(y_test, y_predict)
print("Classification Report:\n")
print(cls_rep)

# Now utilize probabilities instread of predict
y_probs = log.predict_proba(x_test)[:, 1]

# Evaluate across threshhold
threshold = 0.3
y_pred_custom = (y_probs >= threshold).astype(int)
y_pred_custom

# Lower threshold → more positives → fewer misses (good for recall)
# Higher threshold → stricter → fewer false alarms (good for precision)

# Evaluate across multiple threshholds
thresholds = [0.1, 0.3, 0.5, 0.7, 0.9]

for t in thresholds:
    y_pred_t = (y_probs >= t).astype(int)
    print(f"Threshold: {t}")
    print("Precision:", precision_score(y_test, y_pred_t))
    print("Recall:", recall_score(y_test, y_pred_t))
    print("---")

# Precission Recall Curve
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(y_test, y_probs)


# optimize for a metric (f1)
from sklearn.metrics import f1_score

best_t = 0
best_f1 = 0

for t in np.linspace(0,1,100):
    y_pred_t = (y_probs >= t).astype(int)
    f1 = f1_score(y_test, y_pred_t)
    
    if f1 > best_f1:
        best_f1 = f1
        best_t = t

print(best_t, best_f1)



# Results nterpretation
# The model achieves moderate accuracy, but recall is particularly important in this context (detecting diabetes), 
# since false negatives are costly. If recall is low, we may need to adjust the classification threshold or use a different model.
# Accuracy ignores the distribution of errors. In imbalanced datasets, metrics like precision, recall, or AUC are more informative.
# The issue is that accuracy implicitly assumes equal misclassification costs and balanced class priors, which rarely holds in real-world problems.

# Questions

# Q: What is MSE?
# A: It is the mean of the squared errors. It is the difference between the actual and predicted values which are the errors. 
# Those errors are squared to place more weight on huge misses and also to remove negatve differences. A near 0 mse is a good prediction.
# MSE is used for regression; in classification we typically use log-loss or classification metrics like accuracy, precision, and recall

# Q: When is accuracy misleading?
# A: Accuracy ignores the distribution of errors. In imbalanced datasets, metrics like precision, recall, or AUC are more informative

# Q: Difference between precision vs recall?
# A: Precision is the ratio of correct predictions of event occurence to the total predictions of event occurences. 
# Ex. Given the model predicted the event would occur how often was it correct in predictng the actual occurence. 
# Focus: Quality: High precicion means very few false alarms
# A: Recall is the ratio of correct predictions of event occurence to the actual number of event occurences.
# Ex. Gicen that the actual event did occur, how often was the model correct in predicting the occurence
# Focus: Quantity. High recall means the model didn't miss many real cases. 
# There is a tradeoff between precision and recall controlled by the classification threshold.
 

Accuracy Score 0.7402597402597403
Precision Score 0.6666666666666666
Recall Score 0.5185185185185185
Classification Report:

              precision    recall  f1-score   support

           0       0.77      0.86      0.81       150
           1       0.67      0.52      0.58        81

    accuracy                           0.74       231
   macro avg       0.72      0.69      0.70       231
weighted avg       0.73      0.74      0.73       231

Threshold: 0.1
Precision: 0.44886363636363635
Recall: 0.9753086419753086
---
Threshold: 0.3
Precision: 0.6138613861386139
Recall: 0.7654320987654321
---
Threshold: 0.5
Precision: 0.6666666666666666
Recall: 0.5185185185185185
---
Threshold: 0.7
Precision: 0.7837837837837838
Recall: 0.35802469135802467
---
Threshold: 0.9
Precision: 1.0
Recall: 0.1111111111111111
---
0.25252525252525254 0.7106598984771574


In [3]:
# Exercise 3 — Random Forest

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Use previous classification dataset

df = pd.read_csv('datasets/diabetes.csv')

# TODO:
# 1. Train RandomForest
# 2. Compare performance vs LogisticRegression

# split data into x and y and then into train and test
x = df[['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age']]
y = df['Outcome']
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.3, random_state=42, stratify=y)

# Train Model
rfc = RandomForestClassifier(random_state=42)
rfc.fit(x_train, y_train)
log =  LogisticRegression(max_iter=1000)
log.fit(x_train, y_train)

# predict
y_pred_rfc = rfc.predict(x_test)
y_pred_log = log.predict(x_test)

# --- 3. Compare Performance ---
print("--- Random Forest Results ---")
print(classification_report(y_test, y_pred_rfc))

print("\n--- Logistic Regression Results ---")
print(classification_report(y_test, y_pred_log))

# Comparison
# Logistic regression utilizes a straight light (linear) to seperate buyers vs non-buyers, 
# while the segmentation of random forest is non-linear capturing more jagged patters which are closer to realistic behaviours

# Logistic Regression
    # Linear decision boundary
    # Assumes log-odds is linear in features
    # High bias, low variance

# Random Forest
    # Ensemble of decision trees
    # Captures nonlinearities + interactions
    # Lower bias, higher variance (but reduced via bagging)

# Decision trees are sensitive to small changes in the training data because splits are based on discrete thresholds, 
    # which can lead to very different structures. Logistic regression is a smooth, parametric model, 
    # so small changes in data result in small changes in coefficients and predictions, leading to lower variance.
# Random Forest may achieve higher recall because it captures nonlinear relationships, 
    # while Logistic Regression may generalize better if the true relationship is approximately linear.
# Random Forest mitigates overfitting through averaging across trees (bagging), but can still overfit if trees are too deep or if noise is learned.
# Logistic regression needs scaling, while random forest does not
# Tree-based models are invariant to feature scaling, whereas logistic regression is sensitive to it. Without scaling, 
    # comparisons may be biased in favor of Random Forest.
# Uneven feature scales create an ill-conditioned loss surface, where gradients differ significantly across directions. 
    # This causes gradient descent to take inefficient zig-zag steps, slowing convergence.

# Final Take-away
    # Random Forest is likely to outperform Logistic Regression if the relationship between features and outcome is nonlinear. 
    # However, Logistic Regression may perform similarly if the signal is mostly linear and has the advantage of interpretability.

--- Random Forest Results ---
              precision    recall  f1-score   support

           0       0.77      0.87      0.82       150
           1       0.68      0.53      0.60        81

    accuracy                           0.75       231
   macro avg       0.73      0.70      0.71       231
weighted avg       0.74      0.75      0.74       231


--- Logistic Regression Results ---
              precision    recall  f1-score   support

           0       0.77      0.86      0.81       150
           1       0.67      0.52      0.58        81

    accuracy                           0.74       231
   macro avg       0.72      0.69      0.70       231
weighted avg       0.73      0.74      0.73       231



In [4]:
#Exercise 4 — XGBoost

# Use previous classification dataset

df = pd.read_csv('datasets/diabetes.csv')

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

# TODO:
# 1. Train XGBoost model
# 2. Compare vs RandomForest

# split data into x and y and then into train and test
x = df[['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age']]
y = df['Outcome']
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.3, random_state=42, stratify=y)

# Train Model
rfc = RandomForestClassifier(random_state=42)
rfc.fit(x_train, y_train)
xgb =  XGBClassifier()
xgb.fit(x_train, y_train)

# predict
y_pred_rfc = rfc.predict(x_test)
y_pred_xgb = xgb.predict(x_test)

# --- 3. Compare Performance ---
print("--- Random Forest Results ---")
print(classification_report(y_test, y_pred_rfc))

print("\n--- XGBoost Results ---")
print(classification_report(y_test, y_pred_xgb))

# Bagging vs boostng
# Bagging: Parallel voting. It builds many independent trees at the same time using random subsets of the data.
#    Each tree is a bit different, and you average their results to cancel out individual errors and reduce overfitting
# Boosting: learning. It builds one tree, looks at what it got wrong (the residuals), and then builds the next tree specifically to fix those errors. 
#   Creates a loss function based on the errors and ams to minimze that loss function. Each new tree is a specialist in the mistakes of the previous trees

# Why XGBoostdominates Competitions
# Speed and Efficiency: It uses "Parallel Processing" not to build trees, but to find the best way to split the data within a single tree. 
#    It’s significantly faster than older boosting methods.
# Regularization: This is its "secret sauce." XGBoost has built-in and regularization (math that punishes overly complex trees). 
#    This prevents it from "memorizing" the data, making it better at predicting new, unseen data.
# Handling Missing Values: It has a "Sparsity-Aware" algorithm. 
#    If you have NaN or missing values, XGBoost automatically learns which direction (left or right) is best to send those missing values. You don't have to fill them in manually!
# Tree Pruning: Most models stop splitting when they stop seeing a gain. 
#    XGBoost grows the tree to its max depth first and then "prunes" backward. 
#    This finds better splits that might look bad at first but lead to great results later.

# In Random Forest, the trees are like a group of people shouting their guesses at the same time. 
# In XGBoost, it’s more like a relay race where each runner is trying to fix the mistakes of the one before them.


--- Random Forest Results ---
              precision    recall  f1-score   support

           0       0.77      0.87      0.82       150
           1       0.68      0.53      0.60        81

    accuracy                           0.75       231
   macro avg       0.73      0.70      0.71       231
weighted avg       0.74      0.75      0.74       231


--- XGBoost Results ---
              precision    recall  f1-score   support

           0       0.79      0.85      0.82       150
           1       0.68      0.57      0.62        81

    accuracy                           0.75       231
   macro avg       0.73      0.71      0.72       231
weighted avg       0.75      0.75      0.75       231



In [5]:
# Exercise 5 — Pipeline

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# TODO:
# Build pipeline:
# scaler → logistic regression

# Import Data
df = pd.read_csv('datasets/diabetes.csv')

# split data into x and y and then into train and test
x = df[['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age']]
y = df['Outcome']
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.3, random_state=42, stratify=y)

# Create Pipeline
log_pipe = Pipeline([('scaler', StandardScaler()), ('logistic_regression', LogisticRegression())])

# Fit Model
log_pipe.fit(x_train,y_train)
y_pred_log = log_pipe.predict(x_test)

print("\n--- Logistic Results ---")
print(classification_report(y_test, y_pred_log))

# There is no data leakage because scaling after splitting the train and test set prevents "peeking" in the test range
# Also by wrapping this in a pipeline it streamlines the process of scaling and fitting
# The streamlined process also is reproducible as new data that is added can be scaled using the same streamlined process


--- Logistic Results ---
              precision    recall  f1-score   support

           0       0.77      0.87      0.82       150
           1       0.68      0.52      0.59        81

    accuracy                           0.74       231
   macro avg       0.72      0.69      0.70       231
weighted avg       0.74      0.74      0.74       231



In [50]:
# Exercise 6 — Feature Importance

import pandas as pd
importances = xgb.feature_importances_

df_features = pd.DataFrame({'feaures': x.columns,'importance': importances}).sort_values('importance', ascending=False)
df_features

# Glucose, BMI an Age are the most important features
# This makes sense

,feaures,importance
1,Glucose,0.267226
5,BMI,0.138315
7,Age,0.133867
0,Pregnancies,0.110754
6,DiabetesPedigreeFunction,0.095990
4,Insulin,0.091806
3,SkinThickness,0.082008
2,BloodPressure,0.080035


In [6]:
# Exercise 7 — Cross Validation

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# TODO:
# Evaluate models using CV

# Create Pipelines
xgb_pipe= Pipeline([('scaler', StandardScaler()), ('xgb', XGBClassifier(eval_metric='logloss'))])
rfc_pipe= Pipeline([('scaler', StandardScaler()), ('rfc', RandomForestClassifier(random_state=42))])

# Fit and predict both modes
rfc_pipe.fit(x_train,y_train)
xgb_pipe.fit(x_train,y_train)
y_pred_rfc = rfc_pipe.predict(x_test)
y_pred_xgb = xgb_pipe.predict(x_test)

# CV Scores
xgb_cv_scores = cross_val_score(xgb_pipe, x_train, y_train, cv=5)
rfc_cv_scores = cross_val_score(rfc_pipe, x_train, y_train, cv=5)

print(f"XGB Individual Fold Scores: {xgb_cv_scores}")
print(f"XGB Mean Accuracy: {xgb_cv_scores.mean():.2f} (+/- {xgb_cv_scores.std() * 2:.2f})")

print(f"RFC Individual Fold Scores: {rfc_cv_scores}")
print(f"RFC Mean Accuracy: {rfc_cv_scores.mean():.2f} (+/- {rfc_cv_scores.std() * 2:.2f})")

XGB Individual Fold Scores: [0.7037037  0.7037037  0.71962617 0.71962617 0.6728972 ]
XGB Mean Accuracy: 0.70 (+/- 0.03)
RFC Individual Fold Scores: [0.72222222 0.73148148 0.76635514 0.76635514 0.74766355]
RFC Mean Accuracy: 0.75 (+/- 0.04)


In [74]:
# Exxercise 7a Try to enhance via grid search

from sklearn.model_selection import GridSearchCV

# 1. Define the grid of settings you want to try
# Tip: Use 'xgb__' prefix if your model is in a Pipeline
param_grid = {
    'xgb__n_estimators': [50, 100, 200],
    'xgb__max_depth': [3, 5, 7],
    'xgb__learning_rate': [0.01, 0.1, 0.2]
}

# 2. Setup the Search
# This will run 3 * 3 * 3 = 27 different versions of your model
grid_search = GridSearchCV(
    estimator=xgb_pipe,
    param_grid=param_grid,
    cv=5,            # Use 5-fold cross-validation for every combination
    scoring='accuracy',
    n_jobs=-1        # Use all your computer's processors to speed it up
)

# 3. Run the search
grid_search.fit(x_train, y_train)

# 4. Get the best results
print(f"Best Score: {grid_search.best_score_:.4f}")
print(f"Best Settings: {grid_search.best_params_}")


Best Score: 0.7504
Best Settings: {'xgb__learning_rate': 0.01, 'xgb__max_depth': 5, 'xgb__n_estimators': 200}


In [7]:
#Exercise 8 — Model Comparison

# Compare:
#    Logistic Regression
#    Random Forest
#    XGBoost

# Create table:
#    Model | Accuracy | Precision | Recall

# Decide:
#   which model is best
#   why

# Assume you have created a pipeline for log, rfc, and xgb

models = {'Logistic Regression': log_pipe, 'Random Forest Classifier': rfc_pipe, "XGBoost": xgb_pipe}

results=[]

for name, model in models.items():
    results.append({
        'Model Name': name,
        'Accuracy': accuracy_score(y_test,model.predict(x_test)),
        'Precision': precision_score(y_test,model.predict(x_test)),
        'Recall': recall_score(y_test,model.predict(x_test))
        })

df_results = pd.DataFrame(results)
df_results

# Logistic regression preforms the worste accross all three scores
#     This is likely because the ensemble method and the gradient boosting add ore complexity for enhanced predictions
# xgboost preforms slightly better on recall, while random forest performs slightly better on precision, ut the differences are small

,Model Name,Accuracy,Precision,Recall
0,Logistic Regression,0.744589,0.677419,0.518519
1,Random Forest Classifier,0.753247,0.687500,0.543210
2,XGBoost,0.753247,0.676471,0.567901


In [8]:
# Exercise 9a — Data Checks and Cleaning beforew A/B Testing 
import pandas as pd

df = pd.read_csv('datasets/ab_data.csv')
df.head(5)
# Check for N/As
df.isna().sum()

# Check if treated user is in control
tr_ctrl_dups = df.loc[df['group'] == 'treatment', 'user_id'].isin(df.loc[df['group'] == 'control', 'user_id']).any()
print(tr_ctrl_dups)

# Convert IDs to sets and find the intersection size
treated_ids = set(df.loc[df['group'] == 'treatment', 'user_id'])
control_ids = set(df.loc[df['group'] == 'control', 'user_id'])

overlap_count = len(treated_ids & control_ids)
print(f"Number of users in both groups: {overlap_count}")
overlap_ids = treated_ids & control_ids


# This creates a 2x2 grid of Group vs. Landing Page
mismatch_check = pd.crosstab(df['group'], df['landing_page'])
print(mismatch_check)

# Clean the dataset to reomve invalid treatment and controls and remove userids both in the control and treatment
df2 = df[
      ((df['group'] == 'treatment') & (df['landing_page'] == 'new_page') |
      (df['group'] == 'control') & (df['landing_page'] == 'old_page')) &
      (~df['user_id'].isin(overlap_ids))
      ]

# Final check to ensure it worked
print(pd.crosstab(df2['group'], df2['landing_page']))
print(f"Overlap check: {df2.loc[df2['group'] == 'treatment', 'user_id'].isin(df2.loc[df2['group'] == 'control', 'user_id']).any()}")

# Check for duplicates in the treatment group
treatment_dupes = df2[df2['group'] == 'treatment']['user_id'].duplicated().any()
print(treatment_dupes)

# Check for duplicates in the control group
control_dupes = df2.loc[df2['group'] == 'control', 'user_id'].duplicated().any()
print(control_dupes)

# Count the duplicates withn treatment group
tr_dup_cnt = df2.loc[df2['group'] == 'treatment','user_id'].duplicated().sum()
print(tr_dup_cnt)   

# Find all ids duplicated
tr_dups = df2.loc[df2['group'] == 'treatment','user_id'][df2.loc[df2['group'] == 'treatment','user_id'].duplicated()]

print(df.loc[df['user_id'].isin(tr_dups)])

# Drop final duplicates
df2 = df2.drop_duplicates(subset='user_id', keep='first')

# Check duplicates
print(df2['user_id'].duplicated().sum())

True
Number of users in both groups: 1895
landing_page  new_page  old_page
group                           
control           1928    145274
treatment       145311      1965
landing_page  new_page  old_page
group                           
control              0    144300
treatment       144390         0
Overlap check: False
True
False
1
      user_id                   timestamp      group landing_page  converted
1899   773192  2017-01-09 05:37:58.781806  treatment     new_page          0
2893   773192  2017-01-14 02:55:59.590927  treatment     new_page          0
0


In [11]:
# Exercise 9b — A/B Testing 

#Task:
    # Compute the following:
        # conversion rate for control group
        # conversion rate for treatment group

# Return a table with:
    # group | conversion_rate

# compute conversion rate for control group
cr = df2.groupby('group', as_index=False)['converted'].mean()
cr

,group,converted
0,control,0.120305
1,treatment,0.118832


In [13]:
# Exercise 10 — Treatment Effect

# Task:
    # Compute the difference in conversion rates

# Interpret:
    # Is treatment better or worse?
    # By how much?

ate = cr.loc[cr['group'] == 'treatment', 'converted'].item() - cr.loc[cr['group'] == 'control', 'converted'].item()
ate

-0.0014731533420630216

In [18]:
# Exercise 11 — Statistical Significance

# Task:
    # Test whether the difference is statistically significant.

# Constraints:
    # Use a proportion test (Z-test)
    # Use α = 0.05

# Output:
    # test statistic
    # p-value
    # conclusion

import numpy as np
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.proportion import confint_proportions_2indep

# Aggregate everything in one pass
stats = df2.groupby('group')['converted'].agg(['sum', 'count'])

# Extract arrays directly from the resulting table
# This ensures treatment and control are always in the right order
successes = stats.loc[['treatment', 'control'], 'sum'].values
n_obs = stats.loc[['treatment', 'control'], 'count'].values

# Run the test
z_score, p_value = proportions_ztest(successes, n_obs, alternative='larger')
print(z_score, p_value )

# Manually assign the values from your existing arrays
# Index 0 is Treatment, Index 1 is Control
count_trt = successes[0]
n_trt = n_obs[0]
count_ctrl = successes[1]
n_ctrl = n_obs[1]

# NOW the function knows what the variables are
ci_low, ci_high = confint_proportions_2indep(count_trt, n_trt, count_ctrl, n_ctrl, method='wald', alpha=0.05)

print(f"95% Confidence Interval: [{ci_low:.4f}, {ci_high:.4f}]")

    

-1.2197686926091686 0.8887237136325259
95% Confidence Interval: [-0.0038, 0.0009]


In [24]:
# Exercise 12 — Regression-Based Approach

# Task:
    # Estimate the treatment effect using a model.

#Constraints:
    # Use a binary outcome model
    # Treatment must be encoded properly

# Answer:
    # What does the treatment coefficient mean?
    # Does it match your earlier result?

import statsmodels.api as sm

# Create the 'intercept' column (needed for statsmodels OLS)
df2['intercept'] = 1

# Create a dummy variable: 1 for treatment, 0 for control
df2['dummy'] = pd.get_dummies(df2['group'])['treatment'].astype(int)
df2['converted']=df2['converted'].astype(int)

# Fit the model
# Dependent variable (y) is 'converted', Independent (X) are 'intercept' and 'ab_page'
logit_mod = sm.OLS(df2['converted'], df2[['intercept', 'dummy']])
results = logit_mod.fit()

print(results.summary())

                            OLS Regression Results                            
Dep. Variable:              converted   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                     1.488
Date:                Fri, 10 Apr 2026   Prob (F-statistic):              0.223
Time:                        15:12:23   Log-Likelihood:                -84681.
No. Observations:              288689   AIC:                         1.694e+05
Df Residuals:                  288687   BIC:                         1.694e+05
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
intercept      0.1203      0.001    140.851      0.0